# Neuromorph vs Noise: Experiment Results Overview

**Goal**: Compare ANN (frame-based) vs SNN (event-based) robustness to image/event noise on DSEC.

**Setup**:
- Architecture: VGG11-SSD for both ANN and SNN
- Dataset: DSEC subset (8 train sequences, 6 val sequences)
- Test: depth-first on `zurich_city_14_c` with full noise coverage
- ANN input: grayscale frames | SNN input: 2-channel event polarity counts
- SNN: timesteps_per_frame=5, beta=0.9, class-balanced loss (bg_weight=0.25)
- ANN: 31 epochs (best val epoch 15) | SNN raw: 60 epochs (ongoing) | SNN sim: 36 epochs (ongoing)

**Date**: 2026-03-25

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import re

matplotlib.rcParams.update({
    'font.size': 11,
    'figure.figsize': (12, 6),
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

## 1. Training Curves

In [ ]:
# ANN training curves (subset8 balanced grayscale, 31 epochs)
ann_train_raw = """Epoch 0: train=5.1003 (cls=1.9473, loc=3.1530) val=4.1638 (cls=1.2648, loc=2.8990) lr=0.001000
Epoch 1: train=4.3180 (cls=1.2744, loc=3.0436) val=3.9776 (cls=1.1324, loc=2.8452) lr=0.001000
Epoch 2: train=4.1052 (cls=1.2059, loc=2.8993) val=3.8440 (cls=1.1067, loc=2.7373) lr=0.001000
Epoch 3: train=3.7694 (cls=1.1538, loc=2.6156) val=3.5373 (cls=1.0861, loc=2.4513) lr=0.001000
Epoch 4: train=3.5238 (cls=1.1268, loc=2.3969) val=3.4679 (cls=1.1063, loc=2.3616) lr=0.001000
Epoch 5: train=3.3178 (cls=1.0971, loc=2.2207) val=3.2786 (cls=1.0741, loc=2.2044) lr=0.001000
Epoch 6: train=3.1346 (cls=1.0677, loc=2.0669) val=3.4102 (cls=1.1176, loc=2.2925) lr=0.001000
Epoch 7: train=3.0069 (cls=1.0493, loc=1.9576) val=3.3364 (cls=1.0999, loc=2.2365) lr=0.001000
Epoch 8: train=2.8793 (cls=1.0243, loc=1.8550) val=3.2071 (cls=1.1021, loc=2.1050) lr=0.001000
Epoch 9: train=2.7712 (cls=1.0021, loc=1.7691) val=3.1952 (cls=1.1171, loc=2.0781) lr=0.001000
Epoch 10: train=2.6664 (cls=0.9815, loc=1.6848) val=3.2156 (cls=1.1208, loc=2.0948) lr=0.001000
Epoch 11: train=2.5768 (cls=0.9606, loc=1.6162) val=3.2078 (cls=1.1237, loc=2.0840) lr=0.001000
Epoch 12: train=2.4934 (cls=0.9433, loc=1.5501) val=3.1156 (cls=1.1390, loc=1.9766) lr=0.001000
Epoch 13: train=2.4058 (cls=0.9223, loc=1.4835) val=3.1321 (cls=1.1176, loc=2.0145) lr=0.001000
Epoch 14: train=2.3235 (cls=0.9062, loc=1.4172) val=3.1483 (cls=1.1514, loc=1.9970) lr=0.001000
Epoch 15: train=2.2476 (cls=0.8822, loc=1.3654) val=3.1058 (cls=1.1430, loc=1.9628) lr=0.001000
Epoch 16: train=2.1787 (cls=0.8663, loc=1.3125) val=3.1236 (cls=1.1541, loc=1.9695) lr=0.001000
Epoch 17: train=2.1168 (cls=0.8495, loc=1.2673) val=3.2358 (cls=1.1582, loc=2.0776) lr=0.001000
Epoch 18: train=2.0486 (cls=0.8300, loc=1.2186) val=3.1554 (cls=1.1651, loc=1.9903) lr=0.001000
Epoch 19: train=1.9889 (cls=0.8126, loc=1.1763) val=3.1819 (cls=1.1617, loc=2.0202) lr=0.001000
Epoch 20: train=1.9381 (cls=0.7986, loc=1.1395) val=3.2865 (cls=1.2084, loc=2.0781) lr=0.001000
Epoch 21: train=1.8764 (cls=0.7810, loc=1.0955) val=3.3087 (cls=1.1991, loc=2.1097) lr=0.001000
Epoch 22: train=1.8231 (cls=0.7635, loc=1.0596) val=3.1891 (cls=1.1903, loc=1.9988) lr=0.001000
Epoch 23: train=1.7752 (cls=0.7485, loc=1.0267) val=3.2074 (cls=1.2099, loc=1.9975) lr=0.001000
Epoch 24: train=1.7257 (cls=0.7348, loc=0.9909) val=3.2316 (cls=1.2057, loc=2.0258) lr=0.001000
Epoch 25: train=1.6891 (cls=0.7254, loc=0.9637) val=3.1718 (cls=1.2157, loc=1.9561) lr=0.001000
Epoch 26: train=1.6384 (cls=0.7075, loc=0.9309) val=3.2212 (cls=1.2477, loc=1.9735) lr=0.001000
Epoch 27: train=1.6072 (cls=0.6975, loc=0.9098) val=3.2470 (cls=1.2287, loc=2.0182) lr=0.001000
Epoch 28: train=1.5719 (cls=0.6864, loc=0.8855) val=3.2605 (cls=1.2618, loc=1.9987) lr=0.001000
Epoch 29: train=1.5358 (cls=0.6750, loc=0.8607) val=3.2242 (cls=1.2564, loc=1.9677) lr=0.001000
Epoch 30: train=1.5042 (cls=0.6659, loc=0.8383) val=3.2493 (cls=1.2921, loc=1.9572) lr=0.001000"""

# SNN raw events training curves (60 epochs, ongoing)
snn_raw_train_raw = """Epoch 0: train=6.0873 (cls=2.1769, loc=3.9104) val=4.8500 (cls=1.9115, loc=2.9385) lr=0.001000
Epoch 1: train=5.5965 (cls=1.8992, loc=3.6973) val=4.5098 (cls=1.6858, loc=2.8240) lr=0.001000
Epoch 2: train=5.2690 (cls=1.6937, loc=3.5753) val=4.2826 (cls=1.5071, loc=2.7755) lr=0.001000
Epoch 3: train=5.0762 (cls=1.5603, loc=3.5160) val=4.2513 (cls=1.4101, loc=2.8412) lr=0.001000
Epoch 4: train=4.8779 (cls=1.4463, loc=3.4316) val=4.0515 (cls=1.3051, loc=2.7463) lr=0.001000
Epoch 5: train=4.7379 (cls=1.3592, loc=3.3787) val=3.9648 (cls=1.2226, loc=2.7422) lr=0.001000
Epoch 10: train=4.3594 (cls=1.1792, loc=3.1802) val=3.7416 (cls=1.0725, loc=2.6691) lr=0.001000
Epoch 15: train=4.1866 (cls=1.1361, loc=3.0505) val=3.7264 (cls=1.0433, loc=2.6831) lr=0.001000
Epoch 20: train=4.0732 (cls=1.1238, loc=2.9493) val=3.6540 (cls=1.0339, loc=2.6200) lr=0.001000
Epoch 25: train=3.9653 (cls=1.1128, loc=2.8524) val=3.6352 (cls=1.0332, loc=2.6020) lr=0.001000
Epoch 30: train=3.9192 (cls=1.1054, loc=2.8137) val=3.6554 (cls=1.0262, loc=2.6292) lr=0.001000
Epoch 35: train=3.8678 (cls=1.1001, loc=2.7677) val=3.6053 (cls=1.0284, loc=2.5769) lr=0.001000
Epoch 40: train=3.8163 (cls=1.0944, loc=2.7219) val=3.5672 (cls=1.0242, loc=2.5430) lr=0.001000
Epoch 45: train=3.7805 (cls=1.0893, loc=2.6912) val=3.5901 (cls=1.0322, loc=2.5579) lr=0.001000
Epoch 47: train=3.7721 (cls=1.0872, loc=2.6850) val=3.5233 (cls=1.0207, loc=2.5026) lr=0.001000
Epoch 50: train=3.7570 (cls=1.0834, loc=2.6736) val=3.5772 (cls=1.0231, loc=2.5541) lr=0.001000
Epoch 55: train=3.7209 (cls=1.0798, loc=2.6410) val=3.5589 (cls=1.0236, loc=2.5353) lr=0.001000
Epoch 59: train=3.6933 (cls=1.0761, loc=2.6172) val=3.5402 (cls=1.0193, loc=2.5209) lr=0.001000"""

# SNN simulated events training curves (36 epochs, ongoing)
snn_sim_train_raw = """Epoch 0: train=6.0049 (cls=2.1736, loc=3.8312) val=4.7726 (cls=1.9059, loc=2.8667) lr=0.001000
Epoch 1: train=5.2796 (cls=1.8835, loc=3.3961) val=4.5481 (cls=1.6860, loc=2.8621) lr=0.001000
Epoch 2: train=4.9466 (cls=1.6790, loc=3.2676) val=4.1586 (cls=1.5065, loc=2.6521) lr=0.001000
Epoch 5: train=4.4649 (cls=1.3341, loc=3.1308) val=3.8993 (cls=1.2126, loc=2.6867) lr=0.001000
Epoch 10: train=4.1880 (cls=1.1745, loc=3.0135) val=3.6285 (cls=1.0783, loc=2.5502) lr=0.001000
Epoch 15: train=4.0439 (cls=1.1374, loc=2.9065) val=3.5522 (cls=1.0338, loc=2.5183) lr=0.001000
Epoch 20: train=3.9840 (cls=1.1226, loc=2.8614) val=3.5771 (cls=1.0198, loc=2.5573) lr=0.001000
Epoch 25: train=3.9361 (cls=1.1135, loc=2.8226) val=3.4960 (cls=1.0167, loc=2.4793) lr=0.001000
Epoch 30: train=3.8890 (cls=1.1077, loc=2.7813) val=3.5433 (cls=1.0150, loc=2.5284) lr=0.001000
Epoch 35: train=3.8726 (cls=1.1042, loc=2.7684) val=3.5700 (cls=1.0227, loc=2.5473) lr=0.001000"""

def parse_training_log(raw):
    rows = []
    for line in raw.strip().split('\n'):
        m = re.match(
            r'Epoch (\d+): train=([\d.]+) \(cls=([\d.]+), loc=([\d.]+)\) '
            r'val=([\d.]+) \(cls=([\d.]+), loc=([\d.]+)\)',
            line
        )
        if m:
            rows.append({
                'epoch': int(m.group(1)),
                'train_loss': float(m.group(2)),
                'train_cls': float(m.group(3)),
                'train_loc': float(m.group(4)),
                'val_loss': float(m.group(5)),
                'val_cls': float(m.group(6)),
                'val_loc': float(m.group(7)),
            })
    return pd.DataFrame(rows)

ann_curves = parse_training_log(ann_train_raw)
snn_raw_curves = parse_training_log(snn_raw_train_raw)
snn_sim_curves = parse_training_log(snn_sim_train_raw)

print(f"ANN: {len(ann_curves)} epochs, best val={ann_curves['val_loss'].min():.4f} @ epoch {ann_curves.loc[ann_curves['val_loss'].idxmin(), 'epoch']}")
print(f"SNN raw: {len(snn_raw_curves)} epochs, best val={snn_raw_curves['val_loss'].min():.4f} @ epoch {snn_raw_curves.loc[snn_raw_curves['val_loss'].idxmin(), 'epoch']}")
print(f"SNN sim: {len(snn_sim_curves)} epochs, best val={snn_sim_curves['val_loss'].min():.4f} @ epoch {snn_sim_curves.loc[snn_sim_curves['val_loss'].idxmin(), 'epoch']}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Total loss
ax = axes[0]
ax.plot(ann_curves['epoch'], ann_curves['train_loss'], 'b-', label='ANN train', alpha=0.8)
ax.plot(ann_curves['epoch'], ann_curves['val_loss'], 'b--', label='ANN val', alpha=0.8)
ax.plot(snn_raw_curves['epoch'], snn_raw_curves['train_loss'], 'r-', label='SNN raw train', alpha=0.8)
ax.plot(snn_raw_curves['epoch'], snn_raw_curves['val_loss'], 'r--', label='SNN raw val', alpha=0.8)
ax.plot(snn_sim_curves['epoch'], snn_sim_curves['train_loss'], 'g-', label='SNN sim train', alpha=0.8)
ax.plot(snn_sim_curves['epoch'], snn_sim_curves['val_loss'], 'g--', label='SNN sim val', alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Total Loss')
ax.set_title('Training & Validation Loss')
ax.legend(fontsize=8)

# Classification loss
ax = axes[1]
ax.plot(ann_curves['epoch'], ann_curves['train_cls'], 'b-', label='ANN train', alpha=0.8)
ax.plot(ann_curves['epoch'], ann_curves['val_cls'], 'b--', label='ANN val', alpha=0.8)
ax.plot(snn_raw_curves['epoch'], snn_raw_curves['train_cls'], 'r-', label='SNN raw train', alpha=0.8)
ax.plot(snn_raw_curves['epoch'], snn_raw_curves['val_cls'], 'r--', label='SNN raw val', alpha=0.8)
ax.plot(snn_sim_curves['epoch'], snn_sim_curves['train_cls'], 'g-', label='SNN sim train', alpha=0.8)
ax.plot(snn_sim_curves['epoch'], snn_sim_curves['val_cls'], 'g--', label='SNN sim val', alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Classification Loss')
ax.set_title('Classification Loss')
ax.legend(fontsize=8)

# Localization loss
ax = axes[2]
ax.plot(ann_curves['epoch'], ann_curves['train_loc'], 'b-', label='ANN train', alpha=0.8)
ax.plot(ann_curves['epoch'], ann_curves['val_loc'], 'b--', label='ANN val', alpha=0.8)
ax.plot(snn_raw_curves['epoch'], snn_raw_curves['train_loc'], 'r-', label='SNN raw train', alpha=0.8)
ax.plot(snn_raw_curves['epoch'], snn_raw_curves['val_loc'], 'r--', label='SNN raw val', alpha=0.8)
ax.plot(snn_sim_curves['epoch'], snn_sim_curves['train_loc'], 'g-', label='SNN sim train', alpha=0.8)
ax.plot(snn_sim_curves['epoch'], snn_sim_curves['val_loc'], 'g--', label='SNN sim val', alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Localization Loss')
ax.set_title('Localization Loss')
ax.legend(fontsize=8)

fig.suptitle('ANN vs SNN Training Curves (DSEC Subset8 Balanced)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\nKey observation: ANN train loss drops much faster (overfits by epoch ~9).")
print("SNN train/val losses converge slowly and stay close together (underfitting).")
print("SNN cls loss plateaus near 1.08-1.10 -- barely better than random for 9-class softmax.")

## 2. ANN Noise Degradation (Main Result)

Evaluated on `zurich_city_14_c` with latest subset8 balanced grayscale model.
Noise types applied to RGB frames, then evaluated with standard SSD post-processing (conf=0.2, NMS=0.5).

In [ ]:
# Latest ANN eval results (subset8 balanced grayscale, zurich_city_14_c)
ann_results_raw = {
    'clean_s0':            {'mAP50': 0.174374, 'mAP75': 0.052030, 'preds': 56451},
    # Gaussian noise
    'gaussian_noise_s1':   {'mAP50': 0.148211, 'mAP75': 0.027473, 'preds': 159194},
    'gaussian_noise_s2':   {'mAP50': 0.120791, 'mAP75': 0.013054, 'preds': 195123},
    'gaussian_noise_s3':   {'mAP50': 0.087117, 'mAP75': 0.007095, 'preds': 281349},
    'gaussian_noise_s4':   {'mAP50': 0.042343, 'mAP75': 0.001763, 'preds': 245806},
    'gaussian_noise_s5':   {'mAP50': 0.010114, 'mAP75': 0.000426, 'preds': 248894},
    # Shot noise
    'shot_noise_s1':       {'mAP50': 0.163643, 'mAP75': 0.043290, 'preds': 81372},
    'shot_noise_s2':       {'mAP50': 0.154597, 'mAP75': 0.033785, 'preds': 111016},
    'shot_noise_s3':       {'mAP50': 0.129388, 'mAP75': 0.020480, 'preds': 155710},
    'shot_noise_s4':       {'mAP50': 0.102924, 'mAP75': 0.016414, 'preds': 218453},
    'shot_noise_s5':       {'mAP50': 0.080258, 'mAP75': 0.004669, 'preds': 237163},
    # Impulse noise
    'impulse_noise_s1':    {'mAP50': 0.122114, 'mAP75': 0.021985, 'preds': 180189},
    'impulse_noise_s2':    {'mAP50': 0.100324, 'mAP75': 0.005876, 'preds': 274996},
    'impulse_noise_s3':    {'mAP50': 0.090644, 'mAP75': 0.002570, 'preds': 286904},
    'impulse_noise_s4':    {'mAP50': 0.047734, 'mAP75': 0.001349, 'preds': 287721},
    'impulse_noise_s5':    {'mAP50': 0.030322, 'mAP75': 0.000006, 'preds': 266091},
    # Glass blur
    'glass_blur_s1':       {'mAP50': 0.156963, 'mAP75': 0.074236, 'preds': 39714},
    'glass_blur_s2':       {'mAP50': 0.122452, 'mAP75': 0.058565, 'preds': 32166},
    'glass_blur_s3':       {'mAP50': 0.078158, 'mAP75': 0.035693, 'preds': 26487},
    'glass_blur_s4':       {'mAP50': 0.063805, 'mAP75': 0.023541, 'preds': 20771},
    'glass_blur_s5':       {'mAP50': 0.053831, 'mAP75': 0.022638, 'preds': 13439},
    # Motion blur
    'motion_blur_s1':      {'mAP50': 0.166302, 'mAP75': 0.066150, 'preds': 42466},
    'motion_blur_s2':      {'mAP50': 0.144895, 'mAP75': 0.044993, 'preds': 33270},
    'motion_blur_s3':      {'mAP50': 0.120688, 'mAP75': 0.026123, 'preds': 25092},
    'motion_blur_s4':      {'mAP50': 0.097903, 'mAP75': 0.017085, 'preds': 20272},
    'motion_blur_s5':      {'mAP50': 0.067912, 'mAP75': 0.015767, 'preds': 21184},
    # Snow
    'snow_s1':             {'mAP50': 0.105142, 'mAP75': 0.035447, 'preds': 152214},
    'snow_s2':             {'mAP50': 0.001800, 'mAP75': 0.000203, 'preds': 214110},
    'snow_s3':             {'mAP50': 0.028292, 'mAP75': 0.009917, 'preds': 194061},
    'snow_s4':             {'mAP50': 0.019075, 'mAP75': 0.018208, 'preds': 199782},
    'snow_s5':             {'mAP50': 0.000165, 'mAP75': 0.000087, 'preds': 179432},
    # Fog
    'fog_s1':              {'mAP50': 0.000033, 'mAP75': 0.000000, 'preds': 97837},
    'fog_s2':              {'mAP50': 0.011364, 'mAP75': 0.000000, 'preds': 98672},
    'fog_s3':              {'mAP50': 0.000067, 'mAP75': 0.000000, 'preds': 98852},
    'fog_s4':              {'mAP50': 0.000018, 'mAP75': 0.000000, 'preds': 100085},
    'fog_s5':              {'mAP50': 0.000000, 'mAP75': 0.000000, 'preds': 99467},
    # Frost
    'frost_s1':            {'mAP50': 0.050590, 'mAP75': 0.030309, 'preds': 145612},
    'frost_s2':            {'mAP50': 0.000045, 'mAP75': 0.000014, 'preds': 144799},
    'frost_s3':            {'mAP50': 0.000022, 'mAP75': 0.000004, 'preds': 141065},
    'frost_s4':            {'mAP50': 0.000004, 'mAP75': 0.000002, 'preds': 141591},
    'frost_s5':            {'mAP50': 0.030305, 'mAP75': 0.000001, 'preds': 139442},
}

# Parse into structured DataFrame
rows = []
for cond, vals in ann_results_raw.items():
    if cond == 'clean_s0':
        noise_type, severity = 'clean', 0
    else:
        parts = cond.rsplit('_s', 1)
        noise_type = parts[0]
        severity = int(parts[1])
    rows.append({
        'condition': cond,
        'noise_type': noise_type,
        'severity': severity,
        'mAP50': vals['mAP50'],
        'mAP75': vals['mAP75'],
        'num_preds': vals['preds'],
    })

ann_df = pd.DataFrame(rows)
clean_map50 = ann_df.loc[ann_df['noise_type'] == 'clean', 'mAP50'].values[0]
ann_df['relative_mAP50'] = ann_df['mAP50'] / clean_map50

print(f"ANN clean baseline: mAP@0.5 = {clean_map50:.4f}")
print(f"\nConditions evaluated: {len(ann_df)}")
ann_df.sort_values(['noise_type', 'severity'])

In [ ]:
# Noise degradation curves: mAP@0.5 vs severity
noise_types = ['gaussian_noise', 'shot_noise', 'impulse_noise', 'glass_blur',
               'motion_blur', 'snow', 'fog', 'frost']

colors = plt.cm.tab10(np.linspace(0, 1, len(noise_types)))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Absolute mAP@0.5
ax = axes[0]
ax.axhline(y=clean_map50, color='black', linestyle=':', alpha=0.5, label=f'clean ({clean_map50:.3f})')
for noise, color in zip(noise_types, colors):
    subset = ann_df[ann_df['noise_type'] == noise].sort_values('severity')
    ax.plot(subset['severity'], subset['mAP50'], 'o-', color=color, label=noise, markersize=5)
ax.set_xlabel('Severity')
ax.set_ylabel('mAP@0.5')
ax.set_title('ANN Noise Degradation (Absolute)')
ax.legend(fontsize=8, ncol=2)
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_ylim(bottom=-0.01)

# Relative to clean
ax = axes[1]
ax.axhline(y=1.0, color='black', linestyle=':', alpha=0.5, label='clean (1.0)')
for noise, color in zip(noise_types, colors):
    subset = ann_df[ann_df['noise_type'] == noise].sort_values('severity')
    ax.plot(subset['severity'], subset['relative_mAP50'], 'o-', color=color, label=noise, markersize=5)
ax.set_xlabel('Severity')
ax.set_ylabel('Relative mAP@0.5 (vs clean)')
ax.set_title('ANN Noise Degradation (Relative to Clean)')
ax.legend(fontsize=8, ncol=2)
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_ylim(-0.05, 1.15)

plt.tight_layout()
plt.show()

In [ ]:
# Grouped bar chart: mAP@0.5 at severity 1, 3, 5 for each noise type
fig, ax = plt.subplots(figsize=(14, 5))

severities = [1, 3, 5]
x = np.arange(len(noise_types))
width = 0.25
sev_colors = ['#2196F3', '#FF9800', '#F44336']

for i, sev in enumerate(severities):
    vals = []
    for nt in noise_types:
        row = ann_df[(ann_df['noise_type'] == nt) & (ann_df['severity'] == sev)]
        vals.append(row['mAP50'].values[0] if len(row) > 0 else 0)
    ax.bar(x + i * width, vals, width, label=f'Severity {sev}', color=sev_colors[i], alpha=0.85)

ax.axhline(y=clean_map50, color='black', linestyle='--', alpha=0.5, label=f'Clean baseline ({clean_map50:.3f})')
ax.set_xlabel('Noise Type')
ax.set_ylabel('mAP@0.5')
ax.set_title('ANN Detection Performance Under Noise (zurich_city_14_c)')
ax.set_xticks(x + width)
ax.set_xticklabels([n.replace('_', ' ') for n in noise_types], rotation=30, ha='right')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Summary table: relative degradation at severity 3
print("ANN Relative mAP@0.5 at Severity 3 (1.0 = clean baseline)")
print("=" * 55)
for nt in noise_types:
    row = ann_df[(ann_df['noise_type'] == nt) & (ann_df['severity'] == 3)]
    if len(row) > 0:
        rel = row['relative_mAP50'].values[0]
        abs_val = row['mAP50'].values[0]
        print(f"  {nt:20s}  {abs_val:.4f}  ({rel:.1%} of clean)")

print(f"\n  {'clean':20s}  {clean_map50:.4f}  (baseline)")

# Categorize by degradation severity
print("\n\nDegradation categories at severity 3:")
print("-" * 55)
for nt in noise_types:
    row = ann_df[(ann_df['noise_type'] == nt) & (ann_df['severity'] == 3)]
    if len(row) > 0:
        rel = row['relative_mAP50'].values[0]
        if rel > 0.6:
            cat = 'MILD'
        elif rel > 0.2:
            cat = 'MODERATE'
        elif rel > 0.01:
            cat = 'SEVERE'
        else:
            cat = 'CATASTROPHIC'
        print(f"  {nt:20s}  {cat}")

## 3. ANN Per-Class Analysis

In [ ]:
# Per-class AP for ANN clean (latest model)
ann_perclass_clean = {
    'pedestrian': {'ap': 0.1390, 'gt': 1846, 'pred': 5662, 'tp': 422, 'fp': 5240},
    'rider':      {'ap': 0.0000, 'gt': 35,   'pred': 261,  'tp': 0,   'fp': 261},
    'car':        {'ap': 0.2636, 'gt': 3277, 'pred': 45168,'tp': 1740,'fp': 43428},
    'bus':        {'ap': 0.6434, 'gt': 164,  'pred': 3558, 'tp': 145, 'fp': 3413},
    'truck':      {'ap': 0.0000, 'gt': 154,  'pred': 117,  'tp': 0,   'fp': 117},
    'bicycle':    {'ap': 0.0003, 'gt': 92,   'pred': 1618, 'tp': 4,   'fp': 1614},
    'motorcycle': {'ap': 0.0000, 'gt': 0,    'pred': 67,   'tp': 0,   'fp': 67},
    'train':      {'ap': 0.0000, 'gt': 0,    'pred': 0,    'tp': 0,   'fp': 0},
}

pc_df = pd.DataFrame(ann_perclass_clean).T
pc_df.index.name = 'class'

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# AP by class
ax = axes[0]
active = pc_df[pc_df['gt'] > 0]
ax.barh(active.index, active['ap'], color='steelblue')
ax.set_xlabel('AP@0.5')
ax.set_title('ANN Per-Class AP@0.5 (clean)')
ax.set_xlim(0, 0.75)
for i, (idx, row) in enumerate(active.iterrows()):
    ax.text(row['ap'] + 0.01, i, f"{row['ap']:.3f}", va='center', fontsize=9)

# GT distribution
ax = axes[1]
active_gt = active[active['gt'] > 0]
ax.barh(active_gt.index, active_gt['gt'], color='coral')
ax.set_xlabel('Ground Truth Count')
ax.set_title('Ground Truth Distribution (zurich_city_14_c)')
for i, (idx, row) in enumerate(active_gt.iterrows()):
    ax.text(row['gt'] + 30, i, f"{int(row['gt'])}", va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\nPrecision at clean:")
for cls, vals in ann_perclass_clean.items():
    if vals['pred'] > 0:
        prec = vals['tp'] / vals['pred']
        print(f"  {cls:12s}: precision={prec:.3f} ({vals['tp']}/{vals['pred']})")

## 4. SNN Results

SNN evaluated on same test sequence with raw events (events/left/events.h5).
Also tested: simulated events from clean RGB (v2e).

In [ ]:
snn_results = {
    'snn_raw_clean':  {'mAP50': 0.001650, 'mAP75': 0.000700, 'preds': 4022390, 'imgs': 1184, 'gts': 5544},
    'snn_raw_snow_s1':{'mAP50': 0.001650, 'mAP75': 0.000700, 'preds': 4022390, 'imgs': 1184, 'gts': 5544},
    'snn_raw_snow_s3':{'mAP50': 0.001650, 'mAP75': 0.000700, 'preds': 4022390, 'imgs': 1184, 'gts': 5544},
    'snn_raw_snow_s5':{'mAP50': 0.001650, 'mAP75': 0.000700, 'preds': 4022390, 'imgs': 1184, 'gts': 5544},
    'snn_sim_clean':  {'mAP50': 0.000231, 'mAP75': 0.000023, 'preds': 4047074, 'imgs': 1184, 'gts': 5544},
}

print("SNN Evaluation Results (zurich_city_14_c)")
print("=" * 80)
print(f"{'Condition':<20s} {'mAP@0.5':>10s} {'mAP@0.75':>10s} {'Preds':>12s} {'Preds/frame':>12s}")
print("-" * 80)
for cond, vals in snn_results.items():
    pf = vals['preds'] / vals['imgs']
    print(f"{cond:<20s} {vals['mAP50']:>10.6f} {vals['mAP75']:>10.6f} {vals['preds']:>12,d} {pf:>12,.0f}")

print(f"\n\nFor comparison:")
print(f"  ANN clean: mAP@0.5 = {clean_map50:.4f}, ~{56451/1191:.0f} preds/frame")
print(f"  SNN clean: mAP@0.5 = 0.0017, ~{4022390/1184:.0f} preds/frame")
print(f"  Ratio: ANN is {clean_map50/0.001650:.0f}x better at mAP, SNN produces {(4022390/1184)/(56451/1191):.0f}x more predictions")

In [ ]:
# SNN per-class breakdown (raw events, clean)
snn_perclass = {
    'pedestrian': {'ap': 0.0003, 'gt': 1846, 'pred': 46811,  'tp': 62,   'fp': 46749},
    'rider':      {'ap': 0.0000, 'gt': 35,   'pred': 4,      'tp': 0,    'fp': 4},
    'car':        {'ap': 0.0096, 'gt': 3253, 'pred': 3975239,'tp': 1838, 'fp': 3973401},
    'bus':        {'ap': 0.0000, 'gt': 164,  'pred': 279,    'tp': 0,    'fp': 279},
    'truck':      {'ap': 0.0000, 'gt': 154,  'pred': 42,     'tp': 0,    'fp': 42},
    'bicycle':    {'ap': 0.0000, 'gt': 92,   'pred': 15,     'tp': 0,    'fp': 15},
}

print("SNN Per-Class Breakdown (raw events, clean)")
print("=" * 70)
print(f"{'Class':<14s} {'AP@0.5':>8s} {'GT':>6s} {'Pred':>10s} {'TP':>6s} {'FP':>10s} {'Prec':>8s}")
print("-" * 70)
for cls, v in snn_perclass.items():
    prec = v['tp'] / v['pred'] if v['pred'] > 0 else 0
    print(f"{cls:<14s} {v['ap']:>8.4f} {v['gt']:>6d} {v['pred']:>10,d} {v['tp']:>6d} {v['fp']:>10,d} {prec:>8.5f}")

print("\n98.8% of SNN predictions are 'car'. Precision for car: 0.046%.")
print("The SNN has not learned to discriminate background from foreground.")

In [ ]:
# Side-by-side comparison: ANN vs SNN
print("ANN vs SNN Head-to-Head (clean, zurich_city_14_c)")
print("=" * 60)
print(f"{'Metric':<25s} {'ANN':>15s} {'SNN (raw)':>15s}")
print("-" * 60)
metrics = [
    ('mAP@0.5',       f'{clean_map50:.4f}',     '0.0017'),
    ('mAP@0.75',      '0.0520',                  '0.0007'),
    ('Total predictions',  '56,451',              '4,022,390'),
    ('Preds / frame',      '~47',                 '~3,400'),
    ('Car TP',             '1,740',               '1,838'),
    ('Car FP',             '43,428',              '3,973,401'),
    ('Car precision',      '3.85%',               '0.046%'),
    ('Best val loss',      '3.106',               '3.523'),
    ('Training epochs',    '31',                   '60 (ongoing)'),
    ('Input',              'Grayscale frames',     '2ch event counts'),
    ('Conf threshold',     '0.2',                  '0.2'),
    ('At conf=0.5',        'Works',                '0 predictions'),
]
for name, ann_v, snn_v in metrics:
    print(f"  {name:<25s} {ann_v:>15s} {snn_v:>15s}")

print("\nNote: SNN raw event results are IDENTICAL across all RGB noise conditions")
print("because raw events from the event camera are unaffected by RGB corruption.")

## 5. Why SNN Detection Fails: Diagnosis

### Confidence Distribution Problem
- At `conf_threshold=0.2`: SNN outputs ~3,400 detections/frame (mostly "car")
- At `conf_threshold=0.5`: SNN outputs **0** detections
- All SNN softmax confidences are jammed in the 0.2-0.5 band

### Root Causes
1. **Background weight = 0.25 in SSD loss**: The model is penalized 4x less for false positives (misclassifying background as foreground). This biases the model toward predicting non-background everywhere.

2. **Binary spike bottleneck**: LIF neurons produce binary 0/1 spikes. The classification Conv2d heads receive these binary feature maps, severely limiting the information content compared to ANN's continuous-valued features.

3. **Loss only at final timestep**: With `timesteps_per_frame=5`, the first 4 forward passes have no gradient signal. Only the 5th pass gets supervised.

4. **Val loss hides the problem**: SSD loss uses hard negative mining on ~100 of ~30,000 total anchors. A model producing near-uniform predictions can have moderate loss on those 100 anchors while having terrible calibration overall.

## 6. Prediction Count Analysis

In [ ]:
# How noise affects ANN prediction count (indicator of false positive explosion)
fig, ax = plt.subplots(figsize=(14, 5))

clean_preds = 56451
for noise, color in zip(noise_types, colors):
    subset = ann_df[ann_df['noise_type'] == noise].sort_values('severity')
    ax.plot(subset['severity'], subset['num_preds'] / 1000, 'o-', color=color, label=noise, markersize=5)

ax.axhline(y=clean_preds / 1000, color='black', linestyle=':', alpha=0.5, label=f'clean ({clean_preds/1000:.0f}K)')
ax.set_xlabel('Severity')
ax.set_ylabel('Total Predictions (thousands)')
ax.set_title('ANN Prediction Count vs Noise Severity (false positive indicator)')
ax.legend(fontsize=8, ncol=2)
ax.set_xticks([1, 2, 3, 4, 5])
plt.tight_layout()
plt.show()

print("Noise types that cause FP explosion: gaussian, shot, impulse, snow")
print("Noise types that suppress predictions: glass_blur, motion_blur")
print(f"\nSNN for reference: ~4,022K predictions (off the chart) regardless of noise condition")

## 7. Summary Table

In [ ]:
# Build comprehensive summary table
summary_rows = []
for nt in noise_types:
    for sev in [1, 2, 3, 4, 5]:
        row = ann_df[(ann_df['noise_type'] == nt) & (ann_df['severity'] == sev)]
        if len(row) > 0:
            r = row.iloc[0]
            summary_rows.append({
                'Noise Type': nt.replace('_', ' ').title(),
                'Severity': sev,
                'ANN mAP@0.5': f"{r['mAP50']:.4f}",
                'ANN mAP@0.75': f"{r['mAP75']:.4f}",
                'ANN Relative': f"{r['relative_mAP50']:.1%}",
                'ANN Preds': f"{int(r['num_preds']):,}",
            })

summary_df = pd.DataFrame(summary_rows)
print("Complete ANN Noise Evaluation Results")
print(f"Clean baseline: mAP@0.5 = {clean_map50:.4f}, 56,451 predictions, 5,568 ground truths")
print(f"Test sequence: zurich_city_14_c, 1,191 images")
print()
print(summary_df.to_string(index=False))

In [ ]:
# Heatmap: relative degradation
pivot_data = []
for nt in noise_types:
    row_data = {'noise': nt.replace('_', ' ')}
    for sev in [1, 2, 3, 4, 5]:
        row = ann_df[(ann_df['noise_type'] == nt) & (ann_df['severity'] == sev)]
        if len(row) > 0:
            row_data[f's{sev}'] = row.iloc[0]['relative_mAP50']
        else:
            row_data[f's{sev}'] = np.nan
    pivot_data.append(row_data)

hm_df = pd.DataFrame(pivot_data).set_index('noise')

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(hm_df.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(5))
ax.set_xticklabels(['S1', 'S2', 'S3', 'S4', 'S5'])
ax.set_yticks(range(len(hm_df)))
ax.set_yticklabels(hm_df.index)
ax.set_xlabel('Severity')
ax.set_title('ANN Relative mAP@0.5 (green=robust, red=degraded)')

for i in range(len(hm_df)):
    for j in range(5):
        val = hm_df.values[i, j]
        if not np.isnan(val):
            color = 'white' if val < 0.3 else 'black'
            ax.text(j, i, f'{val:.0%}', ha='center', va='center', color=color, fontsize=9)

plt.colorbar(im, label='Relative mAP@0.5')
plt.tight_layout()
plt.show()

## 8. Experiment Inventory

### Active Training Runs (as of 2026-03-25)

| Run | Machine | Epoch | Status | W&B |
|-----|---------|-------|--------|-----|
| ANN subset8 balanced gray | 5de37e9747f2 (42803) | 31/31 | **Complete** | `dsec_subset_ann/2le75b2j` |
| SNN raw events | 7f8d62967447 (14434) | 59/200 | Running (+ 4 duplicate PIDs!) | `dsec_subset_snn_raw/z67r4sq1` |
| SNN sim events (v2e H5) | 4701f6917ed0 (14929) | 35/200 | Running | `dsec_subset_snn_sim` |

### Completed Evaluations

| Model | Conditions | Sequence | Notes |
|-------|-----------|----------|-------|
| ANN subset8 balanced | 41 (8 noise types x 5 severities + clean) | zurich_city_14_c | Full coverage |
| SNN raw (best @ ep47) | clean + snow s1-s5 | zurich_city_14_c | All snow identical (raw events unchanged) |
| SNN raw vs sim | clean only | zurich_city_14_c | Sim events even worse (0.0002 mAP) |

### Key Checkpoints

| Checkpoint | Path | Best Val Loss |
|-----------|------|---------------|
| ANN subset8 balanced | `ann_subset8_balanced_gray_.../vgg11_ssd_ann_dsec_best.pth` | 3.106 |
| SNN raw events | `snn_raw_train_eval_20260321_170306/vgg11_ssd_snn_dsec_best.pth` | 3.523 |
| SNN sim events | `snn_sim_tpf5_host14929_20260322_214922/vgg11_ssd_snn_dsec_best.pth` | 3.496 |

## 9. Narrative for Paper

### What works
- ANN noise degradation curves are clean and tell a clear story
- Gaussian/impulse/shot noise cause progressive degradation
- Blur types (glass, motion) are more gracefully handled
- Fog/frost cause catastrophic failure (near zero mAP even at s1)
- The experimental framework is solid and reproducible

### What doesn't work
- SNN detection performance is near-zero (~0.002 mAP@0.5 vs ANN's 0.17)
- SNN on raw events: noise comparison is meaningless (same input regardless of RGB noise)
- SNN confidence calibration is broken (all confidences in 0.2-0.5 range)

### Honest framing
- This is a methodology/framework contribution
- ANN noise degradation results are the primary finding
- SNN limitations are documented with clear technical explanations
- The gap between SNN on pre-binned TUMTraf events vs raw DSEC events is itself informative
- Future work: membrane readout instead of spikes, event binning strategies, bg_weight tuning